In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

def natural_sort( l ): 
    import re 
    """ Sort the given iterable in the way that humans expect.""" 
    convert = lambda text: int(text) if text.isdigit() else text 
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ] 
    return sorted(l, key = alphanum_key)


In [ ]:
metadata_path = '../../results/config/metadata.qcpass.tsv'
bed_targets_path = "../../config/ag-vampir.bed"
wkdir = "../.."


In [ ]:
# Parameters
metadata_path = "results/config/metadata.tsv"
bed_targets_path = "config/ag-vampir.bed"
wkdir = "/mnt/user_shares/snagi/lstm_scratch/network_scratch/ampseq-agvampir-combined"


### Coverage

In this notebook, we calculate coverage across the whole-genome and specifically at the SNP targets of the amplicon panel. Coverage refers to the number of sequencing reads that align to a specific region of the genome. Higher coverage provides greater confidence in variant calls and helps detect low-frequency alleles.

In [ ]:
import sys
import os
sys.path.append(os.path.join(wkdir, 'workflow/lib'))
import ampseeker as amp

# load panel metadata
metadata = pd.read_csv(metadata_path, sep="\t")

panel_metadata = amp.load_bed(bed_targets_path)
contigs = panel_metadata.contig.unique()


#### Whole-genome

Lets plot coverage across the whole genome to see if we have off-target coverage! Off-target reads occur when primers amplify unintended regions of the genome, resulting in coverage at locations outside the targeted amplicons.

In [ ]:
cov_list = []
for sample_id in metadata['sample_id']:
    cov_df = pd.read_csv(f"{wkdir}/results/coverage/{sample_id}.per-base.bed.gz", sep="\t", header=None)
    cov_df.columns = ["contig", "start", "end", "depth"]
    cov_df = cov_df.assign(sample_id=sample_id).query("contig in @contigs")
    cov_list.append(cov_df)

cov_df = pd.concat(cov_list, axis=0)

total_cov_df = cov_df.groupby(['contig', 'start', 'end']).agg({'depth':'sum'}).reset_index()
total_cov_df = total_cov_df.assign(midpoint=lambda x: ((x.start + x.end)/2).astype(int))


In [ ]:
for idx, contig in enumerate(contigs):
    fig = px.scatter(
            total_cov_df.query("contig == @contig"),
            x='midpoint', 
            y="depth",
            color_discrete_sequence=['lightsalmon'],
            title=contig,
            template='simple_white',
            height=300,
            width=750
        )
    
    amp_lines = panel_metadata.query("contig == @contig")[['start', 'amplicon_id']].drop_duplicates()
    for idx2, row in amp_lines.iterrows():
        fig.add_vline(x=row['start'], line_width=1, line_dash="dash", opacity=0.2)
        
    fig.show()


#### Coverage at each target SNP

Target SNP coverage is a critical quality metric that indicates how well each intended mutation site was amplified. Sufficient coverage is essential for accurate genotyping and allele frequency estimation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

target_covs = []
for sample in metadata.sample_id:
    target_cov = pd.read_csv(f"{wkdir}/results/coverage/{sample}.regions.bed.gz", sep="\t", header=None)
    target_cov = target_cov.assign(sample_id=sample)
    target_covs.append(target_cov)

target_cov_df = pd.concat(target_covs, axis=0)
target_cov_df.columns = ['contig', 'start', 'end', 'amplicon_id', 'depth', 'sample_id']
target_cov_df = target_cov_df.merge(panel_metadata, how='left', on=['contig', 'start', 'end', 'amplicon_id'])
snp_targets_sorted = natural_sort(target_cov_df.amplicon_id.unique().astype(str))


In [ ]:
# target_cov_df = target_cov_df.assign(log_depth=lambda x: np.log2(x.depth))
fig = px.box(target_cov_df, x='amplicon_id', y='depth', color='contig', hover_data=['contig', 'mutation'], width=1000, template='simple_white')
fig.update_xaxes(categoryorder='array', categoryarray= snp_targets_sorted )
fig.show()


#### Coverage by sample

In [ ]:
samples_cov = cov_df.groupby(['sample_id', 'contig']).agg({'depth':'sum'})
fig = px.bar(samples_cov.reset_index(), x='sample_id', y='depth', hover_data='contig', color='contig', width=1000, template='simple_white')
fig.show()


#### Coverage by amplicon and sample

In [ ]:
# pivot to heatmap shape and take mean where we have multiple SNPs in one amplicon 
df = target_cov_df.pivot_table(index='amplicon_id', columns='sample_id', values='depth', aggfunc='mean').fillna(0).astype(int)
# reorder amplicons alphabetical
order = natural_sort(df.index.to_list())
df = df.loc[order]

df.to_excel(f"{wkdir}/results/coverage/amplicon_by_sample_depth.xlsx")

from IPython.display import display, Markdown
display(Markdown(f'<a href="{wkdir}/results/coverage/amplicon_by_sample_depth.xlsx">Amplicon by sample read depth (.xlsx)</a>'))

px.imshow(df, width=1000, height=1000, color_continuous_scale='blues', aspect='auto')
